## Task 1 – Pydantic in General (Domain Modeling & Validation)

We will build a small **Research Assistant backend model** using only Pydantic:

1. **Domain models**
   - `User` – id, email, plan, created_at.
   - `ResearchTask` – topic, priority, due_date, tags, status, budget.
   - (Optionally later) `SourceDocument`, `Insight`.

2. **Validation rules**
   - `email` must be valid (`EmailStr`).
   - `due_date` must be in the future.
   - `priority` is an enum (`LOW`, `MEDIUM`, `HIGH`).
   - `tags` normalized (trim, lowercase, deduplicate).
   - `HIGH` priority tasks must have `budget >= 100`.

3. **Derived / convenience logic**
   - `is_overdue` property.
   - Optional `urgency_score` (derived from priority + due date).

4. **Error handling**
   - Catch `ValidationError` and print human-readable errors.
   - Show examples of invalid payloads.

5. **Serialization / export**
   - Use `.model_dump()` / `.model_dump_json()` (Pydantic v2) to show how models are turned into JSON-friendly structures.

## Task 2 – Pydantic with OpenAI GPT (gpt-4o etc.)

We now extend the same *research assistant* domain and wrap OpenAI calls with strict Pydantic models.

### High-level idea

For each `ResearchTask`, we want to:

1. Generate an **outline** for the research.
2. Summarize **source documents**.
3. Produce final **structured insights** (JSON) with citations and recommended next actions.

### Pydantic's role

Pydantic will be used to define and validate:

1. **LLM configuration & prompts**
   - `LLMConfig` – model name (`gpt-4o`), temperature, max_tokens, etc.
   - `PromptMetadata` – task_id, step_type, attempt, created_at.
   - `PromptTemplate` / `RenderedPrompt` – ensure required variables are provided.

2. **Structured responses from OpenAI**
   - `OutlineSection` – title, bullet_points, priority, estimated_tokens.
   - `GeneratedOutlineResponse` – list of sections + metadata.
   - `DocumentSummary` – key_points, relevance_score [0–1], risks, citations.
   - `GeneratedInsightsResponse` – insights, open_questions, recommended_next_actions.

3. **Error & retry handling**
   - `LLMError` – stage, error_type, message, raw_output, attempt.
   - `LLMCallLog` – prompt + response metadata, success flag, error info.
   - On parse failure, we re-prompt the model to fix the JSON using the validation error message.

4. **Pipeline orchestration**
   - `PipelineStep` (base) and concrete steps: `OutlineStep`, `SummarizationStep`, `InsightsStep`.
   - `PipelineRun` – list of steps, overall status, timestamps, token usage metrics.

All of this remains **framework-free**: we will call the OpenAI SDK directly from Python scripts, and use Pydantic as our schema and guardrail system.

### Task 2 – Detailed Steps

1. **Define request/response schemas** (Pydantic) for:
   - `GenerateOutlineRequest` / `GeneratedOutlineResponse`.
   - `SummarizeDocumentRequest` / `DocumentSummary`.
   - `GenerateInsightsRequest` / `GeneratedInsightsResponse`.

2. **Implement a safe OpenAI wrapper** using Pydantic:
   - Accepts `RenderedPrompt` + `LLMConfig`.
   - Calls OpenAI Chat/Responses API (e.g. `gpt-4o`).
   - Tries to parse JSON output into a given `response_model` (Pydantic).
   - On `ValidationError`, logs the problem and attempts a *repair* call with the error + schema.

3. **Compose a multi-step pipeline** for one `ResearchTask`:
   - Step 1: Call OpenAI to generate an outline (validated with Pydantic).
   - Step 2: For each document, call OpenAI to generate `DocumentSummary`.
   - Step 3: Call OpenAI with all summaries to generate final `GeneratedInsightsResponse`.
   - Step 4: Store a `PipelineRun` object with structured logs and metrics.

4. **Add validation & exception handling everywhere**:
   - Validate all user/task inputs with Pydantic.
   - Wrap OpenAI calls with try/except: detect network errors, API errors, and parsing/validation errors.
   - Surface clear errors back to the CLI.

We will implement these in separate Python modules (e.g. `llm_schemas.py`, `llm_client.py`, `pipeline.py`) in the same folder.